In [ ]:
from pathlib import Path
import sys

# Bootstrap local src import path for repo and review-bundle layouts.
_CWD = Path.cwd().resolve()
for _candidate in [_CWD, *_CWD.parents]:
    _src = _candidate / "src"
    if _src.is_dir():
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from dx_chat_entropy.paths import find_repo_root, get_paths

REPO_ROOT = find_repo_root(Path.cwd())
PATHS = get_paths(REPO_ROOT)


# One-vs-Rest LR Estimation

Estimate likelihood ratios (LRs) for every (finding x condition) pair in a
single-sheet Excel workbook.

Supports two granularities:
- **Category-level**: conditions grouped into categories (e.g. "cardiac")
- **Disease-level**: individual diseases (e.g. "acute coronary syndrome")

And two variants each:
- **Plain**: findings as-is from the spreadsheet
- **Plus-minus**: each finding expanded into "Patient has:" / "Patient does not have:" pairs

Workbook layout assumed:
- Sheet 0 only
- Row 0: condition/category headers (cell A1 may be blank or "Diagnosis:")
- Column 0 (rows >= 1): findings
- All other cells start blank and will be filled with LR estimates

In [ ]:
from __future__ import annotations

import os
import pandas as pd
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────

# Category-level workbooks
CAT_INPUT       = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category.xlsx")
CAT_OUTPUT      = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category_filled.xlsx")
CAT_PM_INPUT    = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category_for_plusminus.xlsx")
CAT_PM_OUTPUT   = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category_plusminus_filled.xlsx")

# Disease-level workbooks
DIS_INPUT       = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease.xlsx")
DIS_OUTPUT      = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_filled.xlsx")
DIS_PM_INPUT    = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_for_plusminus.xlsx")
DIS_PM_OUTPUT   = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_plusminus_filled.xlsx")

# Model(s) to use — pass a list; >1 model produces one sheet per model.
model_names = ["gpt-4.1-nano-2025-04-14"]

# Reasoning depth for o-series models (ignored by non-reasoning models).
REASONING_EFFORT = "medium"   # "low" | "medium" | "high"


In [ ]:
"""
Estimate likelihood‑ratios (LRs) for every (finding × disease‑category) pair in a
single‑sheet Excel workbook whose first column lists findings and whose first row
lists the disease categories.

Input  workbook example   :  est_lrs_by_category.xlsx
Output workbook produced  :  est_lrs_by_category_filled.xlsx
───────────────────────────────────────────────────────────────────────────────
Layout assumed
─────────────
• Sheet #0 is the *only* sheet that will be processed.
• Row 0 (index 0): disease‑category targets.  
  – Cell A1 may read “Diagnosis:” or be blank; all other non‑blank cells in
    that first row are treated as separate prediction targets.

• Column 0 (index 0), rows ≥ 1: findings (free‑text strings beginning with
  “Patient Has:” / “Patient Does Not Have:”, etc.).
  – All other cells initially *empty* → will be populated with LR estimates.

The script preserves the original layout and simply fills the empty cells with
floating‑point LR values (or “ERROR” if the LLM call fails).

Multiple models?
────────────────
`model_names` can contain one or many model IDs.  
If you list more than one model, each model gets its *own* sheet in the output
workbook, all named after the model ID.  (The original sheet is left unchanged
in that case.)

Dependencies
────────────
openai >= 1.0,  pydantic >= 2,  pandas,  openpyxl
"""

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION  ───────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category.xlsx")
OUTPUT_FILE = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_category_filled.xlsx")

# Provide one or many model IDs here.
model_names = ["gpt-4.1-nano-2025-04-14"]

# Optional: tune reasoning depth for o‑series models.
REASONING_EFFORT = "medium"   # "low" | "medium" | "high"


# ──────────────────────────────────────────────────────────────────────────────
# 2.  RESPONSE SCHEMA  ────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────

class LRResponse(BaseModel):
    value: float


# ──────────────────────────────────────────────────────────────────────────────
# 3.  LLM CALL  ───────────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────

LR_PROMPT_TEMPLATE = """You are an expert in medical diagnosis whose task is
to estimate a *likelihood ratio* (LR) for a finding with respect to a category of conditions.

You are an expert in medical diagnosis who is giving assessments of how important a piece of information 
is when determining whether a patient has a particularly type of condition condition. 

Your task is to estimate the likelihood ratio of a finding for a category of disease. 

Recall that the likelihood ratio represents how much the ratio between the odds of disease given a result 
for a lab value, whether a physical exam finding is present, or whether a comorbidity is present over the 
odds of disease when you did not know the result.

You will receive inputs in the following format: 
Condition: <A category of interest, followed by the specific diseases that make up that category, e.g. "cardiac">. 
Finding: <piece of information, e.g. ‘Patient does not have: radiation to the neck, arm, or jaw’>.

So, for example. If the odds of a Condition Z being present was 1 (meaning 50% probability) before we knew anything, 
but then we got a result (Finding A) it became 2 (meaning 2:1 odds or 66% probability), 
then the likelihood ratio would be 2. 

Given a category of diseases and a finding, you will provide your best estimate of the likelihood ratio as a floating point number. 
Return your answer in valid JSON with the following schema: { 'value': <floating point number greater than 0> }.\n\n

Remember, stronger evidence in favor of a condition has a value farther above 1. 
Strong evidence against a diagnosis has a value farther below 1 (closer to 0). 
A likelihood ratio of 10 is equally strong evidence for a condition as a likelihood ratio of 0.1 is against it. 
Likelihood ratios near 1 represent weak evidence for or against. 
If the "patient does not have: " some feature that is almost always present, that is strong evidence against.
(pay attention for double negatives- Patient has: no tobacco and Patient does not have: tobacco are identical)

Here is how I would like you to approach the problem:
First, consider the category of diseases you are predicting (Condition: ___). 
Is the condition a medical diagnosis? 
If so, what kind of findings are usually present in someone who has that condition. 
Does the condition specify a certain type of patient? 
If so, how does that change things? 
Then, consider the finding. 
If a finding is much more common among patients who have the condition of interest than among patients who do not have the condition of interest, 
then the likelihood ratio should be high. 
This might be because the finding is a consequence of the disease, 
indicates that an enabling condition is present, 
indicates that a frequently comorbid condition is present, 
or is related to the pathology of the condition. 

In general, likelihood ratios over about 20 are pathognomonic, 
above 5 or so is extremely strong evidence in favor, 
above 2.5 or so is strong evidence, 
above 1.4 is so-so evidence, 
and 1-1.4 is pretty weak evidence. 

Conversely, if the finding is more common in people who do NOT have the condition, 
then the likelihood ratio should be below 1. 

Similarly, a likelihood ratio below 0.05 would exclude the condition in most situations, 
below 0.2 would be extremely strong evidence against, 
below 0.4 would be strong evidence against, 
below 0.71 is so-so, 
and between 0.71 and 1 is pretty weak evidence against (meaning, it just doesn’t change the odds of the condition much). 

Return JSON: {{ "value": <float >0 }}  (no additional keys).
Reply *only* with the JSON object.
"""

def estimate_lr(
    condition: str,
    finding: str,
    client: OpenAI,
    model: str,
) -> float | str:
    """Call the LLM and return a floating‑point LR (or the string 'ERROR')."""
    sys_msgs = [{"role": "system", "content": LR_PROMPT_TEMPLATE}]
    user_msg = {"role": "user",
                "content": f"Condition: {condition}\nFinding: {finding}"}

    # Build request kwargs.
    kwargs: dict = {}
    if model.startswith("o"):          # o3, o3‑mini, o4‑mini, o4, etc.
        kwargs["reasoning_effort"] = REASONING_EFFORT

    print(f"LLM call → model='{model}' | condition='{condition}' | finding='{finding}'")

    try:
        completion = client.beta.chat.completions.parse(
            model=model,
            messages=[*sys_msgs, user_msg],
            response_format=LRResponse,
            **kwargs,
        )
        return completion.choices[0].message.parsed.value
    except Exception as exc:           # catch any parsing / API error
        print(f"LLM error ({model}): {exc}")
        return "ERROR"


# ──────────────────────────────────────────────────────────────────────────────
# 4.  MAIN MATRIX‑FILLING LOGIC  ──────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────────────────────

def fill_matrix(df: pd.DataFrame, model: str, client: OpenAI) -> pd.DataFrame:
    """Return a *copy* of `df` with all blank data cells filled for one model."""
    df_filled = df.copy(deep=True).astype(object)

    # Identify disease categories (non‑NaN entries) in the first row, skipping col 0.
    categories: dict[int, str] = {col: str(df.iloc[0, col]).strip()
                                  for col in range(1, df.shape[1])
                                  if pd.notna(df.iloc[0, col]) and str(df.iloc[0, col]).strip()}
    if not categories:
        raise ValueError("No disease categories detected in row 0.")

    # Iterate over findings (rows ≥ 1)
    for row in range(1, df.shape[0]):
        finding = df.iloc[row, 0]
        if pd.isna(finding) or not str(finding).strip():
            # Skip completely empty rows
            continue

        for col, diagnosis in categories.items():
            # Only fill previously blank / NaN cells.
            if pd.isna(df_filled.iloc[row, col]) or df_filled.iloc[row, col] == "":
                lr = estimate_lr(diagnosis, finding, client, model)
                df_filled.iat[row, col] = lr

    return df_filled


## Scripts to perform the data processing for Cory's July/2025 reasoning request


### Reasoning about categories

"can you run the LRs for each grouping (second attachment) and add LRs to a few rows to round out the addition of a pulmonary grouping (see rows 91-94)? All additions of diagnoses or key features are highlighted in yellow. First few rows of second attachment include sub-groupings but guessing this is getting too complex for this pilot stage." 

#### Non plus_minus version

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Category-level LRs (non-plus-minus)
# ──────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = CAT_INPUT
OUTPUT_FILE = CAT_OUTPUT

def main() -> None:
    df = pd.read_excel(INPUT_FILE, sheet_name=0, header=None)
    df = df.loc[:, ~(df.replace("", pd.NA).isna().all())]

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        if len(model_names) == 1:
            filled = fill_matrix(df, model_names[0], client)
            filled.to_excel(writer, sheet_name="LRs", index=False, header=False)
        else:
            for model in model_names:
                filled = fill_matrix(df, model, client)
                filled.to_excel(writer, sheet_name=model, index=False, header=False)

    print(f"Completed workbook saved -> {OUTPUT_FILE}")

main()

#### Plus-Minus Categories

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Category-level LRs (plus-minus)
# ──────────────────────────────────────────────────────────────────────────────

def build_plusminus_df(df: pd.DataFrame) -> pd.DataFrame:
    """Return a new dataframe where every finding is duplicated into
       'Patient has:' and 'Patient does not have:' variants."""
    header = df.iloc[0].copy()
    findings = df.iloc[1:, 0].dropna().astype(str).str.strip()

    pm_rows = []
    for f in findings:
        low = f.lower()
        if low.startswith("patient has:") or low.startswith("patient does not have:"):
            pm_rows.append(f)
        else:
            pm_rows.append(f"Patient has: {f}")
            pm_rows.append(f"Patient does not have: {f}")

    df_pm = pd.DataFrame(
        "",
        index=range(len(pm_rows) + 1),
        columns=df.columns,
    )
    df_pm.iloc[0] = header
    df_pm.iloc[1:, 0] = pm_rows

    return df_pm.loc[:, ~(df_pm.replace("", pd.NA).isna().all())]


df_orig = pd.read_excel(CAT_PM_INPUT, sheet_name=0, header=None)
df_plusminus = build_plusminus_df(df_orig)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with pd.ExcelWriter(CAT_PM_OUTPUT, engine="openpyxl") as writer:
    if len(model_names) == 1:
        fill_matrix(df_plusminus, model_names[0], client) \
            .to_excel(writer, sheet_name="LRs", index=False, header=False)
    else:
        for m in model_names:
            fill_matrix(df_plusminus, m, client) \
                .to_excel(writer, sheet_name=m, index=False, header=False)

print(f"Plus-minus (categories) workbook saved -> {CAT_PM_OUTPUT}")

### Diseases

#### Non plus-minus version

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Disease-level LRs (non-plus-minus)
# ──────────────────────────────────────────────────────────────────────────────
# NOTE: The disease-level prompt is slightly different from the category prompt.
# We redefine estimate_lr and fill_matrix here with the disease-specific prompt.
"""
Fill a (finding × disease) matrix with LLM‑estimated likelihood ratios (LRs).

• Workbook layout ----------------------------------------
  ┌───────────┬──────────────┬─────────────┬───┐
  │           │  Col‑1       │  Col‑2      │ … │
  │ Row‑0     │ Disease #1   │ Disease #2  │ … │ ← targets
  │ Row‑1…N   │ Finding 1    │             │   │
  │           │ Finding 2    │             │   │
  └───────────┴──────────────┴─────────────┴───┘
  – Sheet 0 only.  
  – Column‑0 (rows ≥1) holds findings; other cells start blank and will be filled.

• Multiple models?  
  – `model_names` may list one or many model IDs.  
  – With >1 model each sheet in the output is named after the model.

Dependencies:  openai  |  pandas  |  pydantic  |  openpyxl
"""

# ─── 1. CONFIGURATION ────────────────────────────────────────────────────────
INPUT_FILE  = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease.xlsx")
OUTPUT_FILE = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_filled.xlsx")

# Provide one or many model IDs here.
model_names = ["gpt-4.1-nano-2025-04-14"]

# Optional: tune reasoning depth for o‑series models.
REASONING_EFFORT = "low"   # "low" | "medium" | "high"


# ─── 2. RESPONSE SCHEMA ──────────────────────────────────────────────────────
class LRResponse(BaseModel):
    value: float

# ─── 3. PROMPT ───────────────────────────────────────────────────────────────
LR_PROMPT_TEMPLATE = """You are an expert in medical diagnosis who is giving assessments of how important a piece of information is when determining whether a patient has a particularly condition. 
Your task is to estimate the likelihood ratio of a finding for a disease. 
Recall that the likelihood ratio represents how much the ratio between the odds of disease given a result for a lab value, whether a physical exam finding is present, or whether a comorbidity is present over the odds of disease when you did not know the result.
You will receive inputs in the following format; Target condition: <Condition, e.g. Patient Has: Cardiac chest pain>. Finding: <piece of information, e.g. ‘Patient does not have: radiation to the neck, arm, or jaw’>.
So, for example. If the odds of a Condition Z being present was 1 (meaning 50% probability) before we knew anything, but then we got a result (Finding A) it became 2 (meaning 2:1 odds or 66% probability), then the likelihood ratio would be 2. 
Given a condition and a finding, you will provide your best estimate of the likelihood ratio as a floating point number. Return your answer in valid JSON with the following schema: { 'value': <floating point number greater than 0> }.\n\n

Remember, stronger evidence in favor of a condition has a value farther above 1. Strong evidence against a diagnosis has a value farther below 1 (closer to 0). A likelihood ratio of 10 is equally strong evidence for a condition as a likelihood ratio of 0.1 is against it. Likelihood ratios near 1 represent weak evidence for or against. 
And if the "patient does not have: " some feature that is almost always present, that is strong evidence against.
(pay attention for double negatives- 'Patient has: no tobacco use' and 'Patient does not have: tobacco use' are identical)

Here is how I would like you to approach the problem:
First, consider the condition you are predicting (Condition: ___). 
Is the condition a medical diagnosis? If so, what kind of findings are usually present in someone who has that condition. 
Does the condition specify a certain type of patient? If so, how does that change things? 
Then, consider the finding. 
If a finding is much more common among patients who have the condition of interest than among patients who do not have the condition of interest, then the likelihood ratio should be high. 
This might be because the finding is a consequence of the disease, indicates that an enabling condition is present, indicates that a frequently comorbid condition is present, or is related to the pathology of the condition. 
In general, likelihood ratios over about 20 are pathognomonic, above 5 or so is extremely strong evidence in favor, above 2.5 or so is strong evidence, above 1.4 is so-so evidence, and 1-1.4 is pretty weak evidence. 
Conversely, if the finding is more common in people who do NOT have the condition, then the likelihood ratio should be below 1. 
Similarly, a likelihood ratio below 0.05 would exclude the condition in most situations, below 0.2 would be extremely strong evidence against, below 0.4 would be strong evidence against, below 0.71 is so-so, and between 0.71 and 1 is pretty weak evidence against (meaning, it just doesn’t change the odds of the condition much). 

Here are some hypothetical examples to consider: 
    Prompt = Target condition: Cardiac Chest Pain. Finding: Patient has: Pain not worse with exertion (requires they clarify exercise 1hr after meal).
    You would reason that because cardiac chest pain is usually worse with exertion because exertion worsens cardiac demand for oxygen, and thus worsens ischemia.
    Response = {
        ‘value’: 0.4
    }

    Prompt =  Target condition: Cardiac Chest Pain. Finding: Patient does not have: tobacco.
    You would reason that because being someone who smokes increases your risk of coronary artery disease, and thus being a never smoker means you’re at less risk… but many people who have heart attacks still smoke, so it’s only a weak predictor. 
    Response = {
        ‘value’: 0.75
    }

    Prompt = Target condition: Cardaic Chest Pain. Finding = Patient has: enjoys playing chess.
    You would reason that because enjoying chest has no relationship to having a heart attack.
    Response = {
        ‘value’: 1
    }

    Prompt = Target condition: Cardiac Chest Pain. Finding = Patient has: pain located behind the sternum
    You would reason that because cardiac chest pain is often experienced behind the sternum (thus, more likely), but so are many other causes of chest pain - like GERD.
    Response = {
        ‘value’: 1.2
    }

    Prompt = Condition: Cardiac Chest Pain. Finding = patient has: pain worse with exertion.
    You would reason that because the increased myocardial oxygen consumption worsens the pain if oxygen delivery to the myocardium is the cause, as it is in heart attacks.
    Response = {
        ‘value’: 3.4
    }

    OK: here’s the prompt.. """

# ─── 4. LLM CALL ──────────────────────────────────────────────────────────────
def estimate_lr(
    disease: str,
    finding: str,
    client: OpenAI,
    model: str,
) -> float | str:
    """Return LR or 'ERROR'."""
    sys_msgs = [{"role": "system", "content": LR_PROMPT_TEMPLATE}]
    user_msg = {"role": "user",
                "content": f"Condition: {disease}\nFinding: {finding}"}
    kwargs: dict = {}
    if model.startswith("o"):
        kwargs["reasoning_effort"] = REASONING_EFFORT

    print(f"LLM call → model='{model}' | disease='{disease}' | finding='{finding}'")

    try:
        completion = client.beta.chat.completions.parse(
            model=model,
            messages=[*sys_msgs, user_msg],
            response_format=LRResponse,
            **kwargs,
        )
        return completion.choices[0].message.parsed.value
    except Exception as exc:
        print(f"LLM error ({model}): {exc}")
        return "ERROR"

# ─── 5. MATRIX FILL ──────────────────────────────────────────────────────────
def fill_matrix(df: pd.DataFrame, model: str, client: OpenAI) -> pd.DataFrame:
    """Return copy of df with blank cells filled for one model."""
    # Trim truly empty columns first (Excel “used‑range” artefacts)
    df = df.loc[:, ~(df.replace("", pd.NA).isna().all())].copy().astype(object)

    # Identify diseases in row‑0 (skip col‑0).
    diseases = {
        col: str(df.iloc[0, col]).strip()
        for col in range(1, df.shape[1])
        if pd.notna(df.iloc[0, col]) and str(df.iloc[0, col]).strip()
    }
    if not diseases:
        raise ValueError("No disease headers found in row 0.")

    # Iterate through findings (rows ≥1) × diseases.
    for row in range(1, df.shape[0]):
        finding = df.iloc[row, 0]
        if pd.isna(finding) or not str(finding).strip():
            continue                                          # skip blank rows
        for col, disease in diseases.items():
            if pd.isna(df.iat[row, col]) or df.iat[row, col] == "":
                lr = estimate_lr(disease, finding, client, model)
                df.iat[row, col] = lr
    return df

In [ ]:
# ─── 6. DRIVER ───────────────────────────────────────────────────────────────
def main() -> None:
    df0 = pd.read_excel(INPUT_FILE, sheet_name=0, header=None)

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        if len(model_names) == 1:
            fill_matrix(df0, model_names[0], client) \
                .to_excel(writer, sheet_name="LRs", index=False, header=False)
        else:
            for m in model_names:
                fill_matrix(df0, m, client) \
                    .to_excel(writer, sheet_name=m, index=False, header=False)
    print(f"Completed workbook saved → {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

## Plus-Minus Diseases

In [ ]:
# ---------------------------------------------------------------------------
# PLUS‑MINUS  D I S E A S E S
# ---------------------------------------------------------------------------
import os
import pandas as pd
from openai import OpenAI

# -------------------------------------------------------------------------
INPUT_PM_DIS  = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_for_plusminus.xlsx")
OUTPUT_PM_DIS = str(REPO_ROOT / "archive/legacy_runs/lr_estimation_2025_07_21/est_lrs_by_disease_plusminus_filled.xlsx")

# Uses the `model_names`, `estimate_lr`, and `fill_matrix`
# from the “Reasoning about Diseases” section.
# -------------------------------------------------------------------------

def build_plusminus_df(df: pd.DataFrame) -> pd.DataFrame:
    """Duplicate every finding into 'has' / 'does not have' variants."""
    header = df.iloc[0].copy()
    findings = df.iloc[1:, 0].dropna().astype(str).str.strip()

    pm_rows = []
    for f in findings:
        low = f.lower()
        if low.startswith("patient has:") or low.startswith("patient does not have:"):
            pm_rows.append(f)
        else:
            pm_rows.append(f"Patient has: {f}")
            pm_rows.append(f"Patient does not have: {f}")

    df_pm = pd.DataFrame(
        "",
        index=range(len(pm_rows) + 1),
        columns=df.columns,
    )
    df_pm.iloc[0] = header
    df_pm.iloc[1:, 0] = pm_rows

    return df_pm.loc[:, ~(df_pm.replace("", pd.NA).isna().all())]


# ––– LOAD, EXPAND, RUN –––––––––––––––––––––––––––––––––––––––––––––––––––––––
df_orig = pd.read_excel(INPUT_PM_DIS, sheet_name=0, header=None)
df_plusminus = build_plusminus_df(df_orig)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with pd.ExcelWriter(OUTPUT_PM_DIS, engine="openpyxl") as writer:
    if len(model_names) == 1:
        fill_matrix(df_plusminus, model_names[0], client) \
            .to_excel(writer, sheet_name="LRs", index=False, header=False)
    else:
        for m in model_names:
            fill_matrix(df_plusminus, m, client) \
                .to_excel(writer, sheet_name=m, index=False, header=False)

print(f"✔  Plus‑minus (diseases) workbook saved → {OUTPUT_PM_DIS}")